# Faz 3 · B3.1 — Colab ortamı

**Sahip:** Kişi B · **Model:** Qwen2.5-1.5B-Instruct + LoRA (Unsloth)

Bu defter eğitim yapmaz. Eğitimin **koşabileceği ortamı kurar ve kanıtlar.**
Boru hattı provası B3.2'de, gerçek eğitim B3.3'te.

---

## Ücretsiz Colab'ın üç kısıtı

Bunlar aksaklık değil, **tasarımı belirleyen şartlar**:

| kısıt | sonucu |
|---|---|
| Tek T4 GPU | ~55k örnek için 4-7 saat |
| Oturum ~90 dk hareketsizlikte, ~12 saatte kesin kopar | 4-7 saatlik tek koşu **büyük olasılıkla yarıda kesilir** |
| `/content` uçucu | Oturum kapanınca dosyalar gider |

Bu yüzden eğitim **tek koşu değil, kaldığı yerden devam edebilen** bir süreç
olarak kurulacak ve her şey **Drive'da** duracak. Tercih değil, zorunluluk.


## 1. GPU var mı

`Runtime → Change runtime type → T4 GPU` seçili olmalı.


In [ ]:
!nvidia-smi


In [ ]:
import torch

assert torch.cuda.is_available(), 'GPU yok! Runtime > Change runtime type > T4 GPU'

ad = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU  : {ad}')
print(f'VRAM : {vram:.1f} GB')
if 'T4' not in ad:
    print(f'{chr(10)}NOT: T4 bekleniyordu, {ad} bulundu. Sureler farkli cikar.')


## 2. Drive

Her Colab oturumu sıfırdan başlar; bu hücre **her açılışta** çalıştırılmalı.
Açılan pencereden Google hesabını seç ve izin ver.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 3. Klasör düzeni

```
/content/drive/MyDrive/codifya/
  veri/        <- Kisi A'nin JSONL dosyalari (BIR KEZ yuklenir)
  checkpoint/  <- egitim ara kayitlari (oturum kopunca buradan devam)
  cikti/       <- merge edilmis model + GGUF
```

> ⚠️ **Veriyi her oturumda yeniden yükleme.** 76 MB'lık `gerekce_train.jsonl`
> her oturumda 10-20 dakika yakar ve kopma riskini artırır. Drive'a bir kez koy.


In [ ]:
from pathlib import Path

KOK = Path('/content/drive/MyDrive/codifya')
VERI = KOK / 'veri'
CHECKPOINT = KOK / 'checkpoint'
CIKTI = KOK / 'cikti'

for d in (VERI, CHECKPOINT, CIKTI):
    d.mkdir(parents=True, exist_ok=True)
    print(f'{d!s:45s} hazir')


## 4. Veri yerinde mi

Dosyalar Drive'daki `codifya/veri/` klasörüne elle yüklenir (Drive arayüzünden
sürükle-bırak). Aşağıdaki hücre eksik olanı söyler.


In [ ]:
BEKLENEN = {
    'gerekce_train.jsonl': 40293,
    'gerekce_val.jsonl': None,
    'gerekce_test.jsonl': None,
    'router_train.jsonl': 43798,
    'router_val.jsonl': None,
    'router_test.jsonl': None,
    'golden_set_aday.jsonl': None,
}

eksik = []
for ad_, beklenen in BEKLENEN.items():
    yol = VERI / ad_
    if not yol.exists():
        eksik.append(ad_)
        print(f'  YOK  {ad_}')
        continue
    mb = yol.stat().st_size / 1024**2
    with yol.open(encoding='utf-8') as f:
        n = sum(1 for _ in f)
    uyari = ''
    if beklenen and n != beklenen:
        uyari = f'  <-- {beklenen} bekleniyordu'
    print(f'  VAR  {ad_:24s} {mb:7.1f} MB  {n:6d} satir{uyari}')

if eksik:
    raise SystemExit(f'Eksik dosya: {eksik} -- Drive/codifya/veri/ klasorune yukle.')
print('Veri tamam.')


## 5. Unsloth

Kurulum 2-3 dakika sürer ve **her oturumda tekrarlanır** (Colab sıfırlanıyor).


In [ ]:
!pip install -q unsloth


In [ ]:
from unsloth import FastLanguageModel

print('unsloth hazir:', FastLanguageModel)


## 6. ⭐ İstem biçimi — eğitimden ÖNCE karara bağlandı

Yanlış karar 5 saatlik eğitimi çöpe atar. Bu yüzden biçim **yerelde** kararlaştırıldı,
veri de yerelde dönüştürüldü. Colab'da dönüştürme adımı yok — yüklenen dosyalar
doğrudan eğitime hazır:

```json
{"istem": "VERILER:\nurun: ...\n\nGEREKCE:", "cevap": "Porselen Karo ..."}
```

Dönüştürmeyi yapan betik: `scratchpad/veri_hazirla.py`. Yüklenecek boyut
**103 MB → 34 MB** düştü ve Colab'da patlayacak bir adım azaldı.

---

### Karar 1 — ürün adı isteme KONUYOR

```
Egitim verisi (Kisi A) : gerekcelerin %100'unde urun adi AYNEN geciyor
B2.4 istemi   (Kisi B) : urun adini BILEREK koymuyordu
```

B2.4'te adı çıkarmıştım çünkü **taban model** Türkçe özel adları bozuyordu:
`Astar Boya` → *starboy*, `İzocam` → *isyancı yalıtım levhası*. Guard bunu
yakalayamaz — uydurulan şey sayı değil.

Eğitimde istem ile hedef arasındaki ilişki neyse, model onu öğrenir:

- **İsteme ad koymaz, hedefte ad varsa** → model *yoktan ad uydurmayı* öğrenir.
  "starboy" sorununu ağırlıklara işler. **En kötü seçenek.**
- **İsteme ad koyar, hedefte ad varsa** → model *adı kopyalamayı* öğrenir.
  Taban model beceremiyordu çünkü hiç öğretilmemişti.

Kopyalama davranışı **B3.2'de 100 örnekle sınanacak** — tam eğitimden önce, ucuz.
Model yine ad bozarsa geri dönüş planı: hedeflerden adları ayıklamak (bir kerelik
dönüşüm, Kişi A'nın veriyi yeniden üretmesi gerekmez).

### Karar 2 — eğitilmiş modelin istemi KISA

B2.4'ün istemi taban model içindi; kurallar bloğu ve few-shot örnek içeriyordu.
Eğitilmiş modelde ikisi de gereksiz — davranış ağırlıklara işlenmiş olacak.
Kısa istem = az token = hızlı üretim.

### Guard düzeltmesi (eğitimi doğrudan etkiledi)

Veriyi incelerken B2.5 guard'ında bir hata bulundu: %2 bağıl sınır, küçük
sayıların 2 ondalıkta **doğru** yuvarlanmasını da kesiyordu
(`0,14444 → "0,14"`, bağıl fark %3,08). Bu, sipariş gerekçelerinin **%16,9'unu**
boşuna reddediyordu. Düzeltildi; artık 5.000 hedefin 5.000'i guard'dan geçiyor.


In [ ]:
import json

with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    ornek = json.loads(f.readline())

print('--- ISTEM ---')
print(ornek['istem'])
print()
print('--- CEVAP (hedef) ---')
print(ornek['cevap'])

assert set(ornek) >= {'istem', 'cevap'}, 'Dosya hazir bicimde degil!'
print()
print('Bicim dogru: istem + cevap')


## 7. Bitti sayılır — üç ölçüt

Üçü de yeşilse B3.2'ye geçilir.


In [ ]:
olcutler = []

try:
    olcutler.append((torch.cuda.is_available(), 'GPU baglandi: ' + torch.cuda.get_device_name(0)))
except Exception as e:
    olcutler.append((False, 'GPU yok: ' + str(e)))

try:
    from unsloth import FastLanguageModel

    olcutler.append((KOK.exists(), 'Drive bagli: ' + str(KOK)))
    olcutler.append((True, 'unsloth import edildi'))
except Exception as e:
    olcutler.append((False, 'unsloth: ' + str(e)))

try:
    with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
        k = json.loads(f.readline())
    tamam = bool(k.get('istem')) and bool(k.get('cevap'))
    olcutler.append((tamam, 'veri/ okunabiliyor ve bicim dogru'))
except Exception as e:
    olcutler.append((False, 'veri okunamadi: ' + str(e)))

print('=' * 60)
for ok, mesaj in olcutler:
    print('  ' + ('GECTI' if ok else 'KALDI') + '  ' + mesaj)
print('=' * 60)
print('B3.1 TAMAM' if all(o for o, _ in olcutler) else 'B3.1 EKSIK')


---

# B3.2 · 100 örnekle boru hattı provası

**Bu adım atlanmaz.** Görev dosyasının gerekçesi:

> Tam eğitim 4-7 saat sürüyor ve ücretsiz Colab'da oturum kopabiliyor.
> Hattın 3. saatinde çıkacak bir biçim hatası, hem 3 saati hem bir Colab
> oturumunu yakar.

Amaç **kalite değil, hattın çalıştığını kanıtlamak**. 100 örnekle eğitilen
model kötü olacak — sorun değil. Kanıtlanmak istenen şu zincir:

```
yukle -> egit -> kaydet -> uret -> (B3.4'te) merge -> GGUF -> Ollama
```

### Burada ayrıca sınanan karar

B3.1'de kararlaştırdık: **ürün adı isteme konuyor**, model kopyalamayı
öğrensin diye. Taban model bunu beceremiyordu (`Astar Boya` → *starboy*).
100 örnek öğrenmeye yetmez ama **eğilimi** gösterir.


## 1. Taban modeli yükle


In [ ]:
from unsloth import FastLanguageModel

MODEL = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ = 1024  # gerekce istemleri kisa; daha uzunu bosa VRAM ve sure yakar

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ,
    dtype=None,
    load_in_4bit=True,
)
print('yuklendi:', MODEL)


## 2. LoRA katmanlarını tak

`r=16`, `alpha=32`, hedef: tüm dikkat + MLP projeksiyonları (görev dosyası).
Eğitilen parametre oranı %1'in altında kalmalı — LoRA'nın bütün amacı bu.


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model.print_trainable_parameters()


## 3. 100 örnek hazırla

Eğitim metni `istem + cevap + EOS` olarak birleştiriliyor. **EOS şart**:
olmazsa model cümleyi nerede bitireceğini öğrenmez ve `max_new_tokens`
dolana kadar saçmalar.


In [ ]:
import json

from datasets import Dataset

N = 100

with (VERI / 'gerekce_train.jsonl').open(encoding='utf-8') as f:
    kayitlar = [json.loads(satir) for _, satir in zip(range(N), f, strict=False)]

metinler = [k['istem'] + '\n' + k['cevap'] + tokenizer.eos_token for k in kayitlar]
ds = Dataset.from_dict({'text': metinler})

print(len(ds), 'ornek hazir')
print('-' * 60)
print(ds[0]['text'])


## 4. Eğit (~2 dakika)

`max_steps=30` bilinçli: amaç öğrenmek değil, hattın çalıştığını görmek.
Kayıp (`loss`) düşüyorsa eğitim gerçekten oluyor demektir.


In [ ]:
from trl import SFTTrainer

ARGS = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=30,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=5,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=42,
    output_dir=str(CHECKPOINT / 'b32'),
    report_to='none',
)

# TRL surumleri arasinda imza degisti; ikisini de deniyoruz.
try:
    from trl import SFTConfig

    trainer = SFTTrainer(
        model=model,
        train_dataset=ds,
        args=SFTConfig(dataset_text_field='text', max_seq_length=MAX_SEQ, **ARGS),
    )
except (ImportError, TypeError) as e:
    print('SFTConfig kullanilamadi, eski imzaya duseliyor:', type(e).__name__)
    from transformers import TrainingArguments

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=ds,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ,
        args=TrainingArguments(**ARGS),
    )

sonuc = trainer.train()
print(sonuc)


## 5. ⭐ Üret ve oku — asıl sınav

Aynı istemi hem **eğitim öncesi hedefle** hem modelin ürettiğiyle
karşılaştırıyoruz. Bakılacak üç şey:

1. Model `GEREKCE:`den sonra **cümle** kuruyor mu (istem tekrarı değil)?
2. **Ürün adını doğru kopyalıyor mu** — yoksa yine *starboy* mu?
3. Cümle bitiyor mu (EOS öğrenilmiş mi)?


In [ ]:
FastLanguageModel.for_inference(model)

for i in (0, 1, 2):
    istem = kayitlar[i]['istem']
    girdi = tokenizer([istem], return_tensors='pt').to('cuda')
    cikti = model.generate(**girdi, max_new_tokens=120, do_sample=False)
    tam = tokenizer.decode(cikti[0], skip_special_tokens=True)
    uretilen = tam[len(istem):].strip()
    ad = kayitlar[i]['istem'].split('urun: ')[1].split(chr(10))[0]

    print('=' * 70)
    print('URUN   :', ad)
    print('URETTI :', uretilen[:300])
    print('HEDEF  :', kayitlar[i]['cevap'][:300])
    print('ad dogru kopyalandi mi :', ad in uretilen)


## 6. LoRA'yı Drive'a kaydet

Yalnızca adaptör kaydediliyor (~50 MB), tam model değil (~3 GB). B3.4'te
taban modelle birleştirilip GGUF'a çevrilecek.


In [ ]:
HEDEF_DIZIN = CIKTI / 'b32_lora'

model.save_pretrained(str(HEDEF_DIZIN))
tokenizer.save_pretrained(str(HEDEF_DIZIN))

import os

toplam = sum(os.path.getsize(os.path.join(k, d)) for k, _, dl in os.walk(HEDEF_DIZIN) for d in dl)
print('kaydedildi:', HEDEF_DIZIN)
print('boyut     :', round(toplam / 1024**2, 1), 'MB')
print('dosyalar  :', sorted(os.listdir(HEDEF_DIZIN)))


## 7. B3.2 bitti sayılır


In [ ]:
olcutler = [
    (len(ds) == N, str(N) + ' ornek yuklendi'),
    (sonuc.training_loss > 0, 'egitim kostu, kayip: ' + str(round(sonuc.training_loss, 4))),
    (HEDEF_DIZIN.exists(), 'LoRA Drive\'a kaydedildi'),
    (bool(uretilen), 'model metin uretti'),
]

print('=' * 60)
for ok, mesaj in olcutler:
    print('  ' + ('GECTI' if ok else 'KALDI') + '  ' + mesaj)
print('=' * 60)
print('B3.2 TAMAM' if all(o for o, _ in olcutler) else 'B3.2 EKSIK')
print()
print('NOT: kalite BEKLENMIYOR. 100 ornek ogrenmeye yetmez.')
print('     Kanitlanan sey hattin ucdan uca calistigi.')
